<a href="https://colab.research.google.com/github/NerusuSahithi011/NLP/blob/main/lab_assignment15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import Libraries

In [1]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
import numpy as np


Load ELMO Model

In [2]:
elmo = hub.load("https://tfhub.dev/google/elmo/3")

Text Corpus

In [3]:
sentences = ["The bank will not approve the loan",
             "He sat on the river bank"]


Generate Embeddings

In [4]:
embeddings = elmo.signatures['default'](tf.constant(sentences))['elmo']
print(embeddings.shape)

(2, 7, 1024)


Inspect Word Embedding (“bank”)

In [5]:
# Convert sentences into tokens
tokenized = [sentence.split() for sentence in sentences]

# Find index of "bank"
idx1 = tokenized[0].index("bank")
idx2 = tokenized[1].index("bank")

# Extract embeddings
bank_emb_1 = embeddings[0][idx1]
bank_emb_2 = embeddings[1][idx2]

print("First 10 values (Sentence 1):", bank_emb_1[:10])
print("First 10 values (Sentence 2):", bank_emb_2[:10])

First 10 values (Sentence 1): tf.Tensor(
[-0.9086765  -0.36578754 -0.07339765  0.586675    0.22132948 -0.7657321
 -0.11816446 -0.30227882 -0.06137528  0.1695182 ], shape=(10,), dtype=float32)
First 10 values (Sentence 2): tf.Tensor(
[-0.15614763  0.28345656 -0.08914638  0.2069506   0.1524184  -0.2037305
 -0.10812807  0.03747292  0.5214868   0.35352594], shape=(10,), dtype=float32)


Compare Context (Cosine Similarity)

In [6]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim = cosine_similarity(bank_emb_1.numpy(), bank_emb_2.numpy())

print("Similarity between 'bank' meanings:", sim)

Similarity between 'bank' meanings: 0.57813895


Task - II

Text Corpus

In [7]:
sentences = ["The bat is flying",
              "He hit the ball with a bat"]

Generate Embeddings

In [8]:
embeddings = elmo.signatures['default'](tf.constant(sentences))['elmo']
print(embeddings.shape)


(2, 7, 1024)


Inspect Word Embedding (“bat”)

In [11]:
# Convert sentences into tokens
tokenized = [sentence.split() for sentence in sentences]

# Find index of "bat"
idx1 = tokenized[0].index("bat")
idx2 = tokenized[1].index("bat")

# Extract embeddings
bat_emb_1 = embeddings[0][idx1]
bat_emb_2 = embeddings[1][idx2]

print("First 10 values (Sentence 1):", bat_emb_1[:10])
print("First 10 values (Sentence 2):", bat_emb_2[:10])

First 10 values (Sentence 1): tf.Tensor(
[-0.45903727 -0.27516434 -0.26348162 -0.12380227 -0.11530864 -0.45880768
 -0.12675564 -0.12455821 -0.2630463  -0.1839562 ], shape=(10,), dtype=float32)
First 10 values (Sentence 2): tf.Tensor(
[-0.20741455  0.01774421  0.12503807 -0.28234187 -0.23928387  0.04694621
 -0.08645728  0.31607887 -0.13618736 -0.05437148], shape=(10,), dtype=float32)


Compare Context (Cosine Similarity)

In [12]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim = cosine_similarity(bat_emb_1.numpy(), bat_emb_2.numpy())

print("Similarity between 'bat' meanings:", sim)

Similarity between 'bat' meanings: 0.64885396



BERT Model Implementation


Task - I

In [13]:

sentences = [
    "The bat is flying",
    "He hit the ball with a bat"
]


In [14]:
# Preprocessing model
preprocess = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")

# BERT encoder
bert_model = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3")


Preprocess Input

In [15]:

inputs = preprocess(sentences)
print(inputs.keys())


dict_keys(['input_type_ids', 'input_mask', 'input_word_ids'])


Generate Embeddings

In [16]:
outputs = bert_model(inputs)

# Word-level embeddings
embeddings = outputs['sequence_output']

print("Embedding shape:", embeddings.shape)

Embedding shape: (2, 128, 768)


Get Tokens

In [17]:
# Use tokenizer from preprocess model
tokenizer = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")

# Tokenized words (for visualization)
tokens = preprocess(sentences)['input_word_ids']

print(tokens)


tf.Tensor(
[[ 101 1996 7151 2003 3909  102    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0]
 [ 101 2002 2718 1996 3608 2007 1037 7151  102    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    

Extract bat Embeddings

In [18]:
bat_emb_1 = embeddings[0][2]  # "bat" in first sentence
bat_emb_2 = embeddings[1][7]  # "bat" in second sentence
print("First 10 values (Sentence 1):", bat_emb_1[:10])
print("First 10 values (Sentence 2):", bat_emb_2[:10])



First 10 values (Sentence 1): tf.Tensor(
[-6.0759485e-05  2.6044574e-01  1.2677923e-01 -2.8091615e-01
  1.3291722e-02 -4.8430106e-01  7.0899105e-01  7.3061728e-01
  2.0883086e-01 -7.8647327e-01], shape=(10,), dtype=float32)
First 10 values (Sentence 2): tf.Tensor(
[-0.06056874 -0.9072487  -0.06744917 -0.27453542 -0.46633738  0.04165877
  0.23526224  0.35728985  0.9171287  -0.6723436 ], shape=(10,), dtype=float32)


Cosine Similarity

In [19]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim = cosine_similarity(bat_emb_1.numpy(), bat_emb_2.numpy())

print("Similarity between 'bat' meanings:", sim)


Similarity between 'bat' meanings: 0.43427736


Task - II

Text Corpus

In [20]:
sentences = ["The bank will not approve the loan",
             "He sat on the river bank"]


In [21]:
# Preprocessing model
preprocess = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")

# BERT encoder
bert_model = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3")


Preprocess Input

In [22]:
inputs = preprocess(sentences)
print(inputs.keys())


dict_keys(['input_type_ids', 'input_mask', 'input_word_ids'])


Generate Embeddings

In [24]:
outputs = bert_model(inputs)

# Word-level embeddings
embeddings = outputs['sequence_output']

print("Embedding shape:", embeddings.shape)

Embedding shape: (2, 128, 768)


Get Tokens

In [25]:
# Use tokenizer from preprocess model
tokenizer = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")

# Tokenized words (for visualization)
tokens = preprocess(sentences)['input_word_ids']

print(tokens)

tf.Tensor(
[[  101  1996  2924  2097  2025 14300  1996  5414   102     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0]
 [  101  2002  2938  2006  1996  2314  2924   102     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0 

Extract bank Embeddings

In [26]:
bank_emb_1 = embeddings[0][2]  # "bank" in first sentence
bank_emb_2 = embeddings[1][7]  # "bank" in second sentence
print("First 10 values (Sentence 1):", bank_emb_1[:10])
print("First 10 values (Sentence 2):", bank_emb_2[:10])



First 10 values (Sentence 1): tf.Tensor(
[ 0.29043403 -0.32690707  0.4144945   0.29487795  1.2237236   0.18653788
 -0.49868098  0.6392791   0.09137864  0.15223756], shape=(10,), dtype=float32)
First 10 values (Sentence 2): tf.Tensor(
[ 0.8744696   0.02230046 -0.24590221  0.6943074  -0.5228018  -0.58300126
  0.37402233 -0.5297509   0.71300304  0.04163592], shape=(10,), dtype=float32)


Cosine Similarity

In [27]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim = cosine_similarity(bank_emb_1.numpy(), bank_emb_2.numpy())

print("Similarity between 'bat' meanings:", sim)


Similarity between 'bat' meanings: -0.0074031646


Text Classification Using ELMO+Naive Bayes

Import Libraries

In [28]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np

from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

Text Corpus

In [30]:
sentences = [
    "Win money now",
    "Claim your prize",
    "Hello how are you",
    "Let's meet tomorrow",
    "Free lottery ticket",
    "Are you coming today"
]
labels = [1, 1, 0, 0, 1, 0]   # 1 = spam, 0 = normal

Load ELMo Model

In [31]:
elmo = hub.load("https://tfhub.dev/google/elmo/3")

Generate ELMo Embeddings

In [32]:
embeddings = elmo.signatures['default'](tf.constant(sentences))['elmo']

print("Shape:", embeddings.shape)

Shape: (6, 4, 1024)


Convert to Sentence Embeddings

In [33]:
sentence_embeddings = tf.reduce_mean(embeddings, axis=1)

X = sentence_embeddings.numpy()
y = np.array(labels)

print("Feature shape:", X.shape)

Feature shape: (6, 1024)


Train-Test Split

In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


Train Model

In [35]:
model = GaussianNB()
model.fit(X_train, y_train)


GaussianNB()

Model Testing

In [36]:
y_pred = model.predict(X_test)

print("Predictions:", y_pred)
print("Actual:", y_test)


Predictions: [0 0]
Actual: [1 1]


Model Evaluation

In [37]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)


Accuracy: 0.0


Prediction on New Text

In [38]:
new_sentence = ["Congratulations! You won a free ticket"]

new_emb = elmo.signatures['default'](tf.constant(new_sentence))['elmo']
new_emb = tf.reduce_mean(new_emb, axis=1)

prediction = model.predict(new_emb.numpy())

print("Prediction:", "Spam" if prediction[0] == 1 else "Not Spam")


Prediction: Not Spam


Text Classification using BERT+NB

In [39]:
preprocess = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
bert_model = hub.load("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3")


In [40]:
bert_inputs = preprocess(sentences)

In [41]:
bert_outputs = bert_model(bert_inputs)

# Use sentence embedding ([CLS])
bert_features = bert_outputs['pooled_output'].numpy()


In [42]:
X_train, X_test, y_train, y_test = train_test_split(
    bert_features, labels, test_size=0.2, random_state=42
)

nb_bert = GaussianNB()
nb_bert.fit(X_train, y_train)

y_pred_bert = nb_bert.predict(X_test)
acc_bert = accuracy_score(y_test, y_pred_bert)

print("BERT Accuracy:", acc_bert)

BERT Accuracy: 0.0
